# Lab 5: Style it, layer it

Your MVP shows one dataset as dots or shapes. Today your map starts to **answer a question**: one layer coloured by a value, a legend that explains the colours, and a second layer the reader can switch on and off.

- **Layer A (30 minutes, everyone):** count the Vancouver sewers into 1 km squares in Python, then colour the squares by the share of old combined pipes, with a legend and a layer switcher.
- **Layer B (the working state, and the mark):** the same three things on **your own** project, a five-second test with a neighbour, and your map published on GitHub Pages.

This is the **last lab on the sewers**. From Lab 6 on, the sewers are retired and every lab builds on your own project.
- **Layer C:** your page's story, a colour check, and your A3 feature.

Run the import cell first.

## Setup

GeoPandas was installed in Lab 4. If the import below says it cannot find geopandas, remove the `#` from the first line, run the cell, restart the kernel (Restart, at the top of the notebook) and run it again.

In [1]:
# %pip install geopandas
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import os
print("GeoPandas", gpd.__version__, "is ready")

GeoPandas 1.2.0 is ready


## Layer A, Part 1. The pipes, in metres

The same sewer table as Lab 4: every pipe's midpoint as x and y in metres, UTM zone 10N (`EPSG:26910`). This time we **keep** the metres for a while, because the next step needs squares that are exactly 1,000 m wide. Metres are for measuring; degrees are for the web map at the end.

We also add one column, `is_combined`, that is `True` for a combined sewer (the old kind that carries rainwater and wastewater in one pipe, and overflows into the harbour in heavy rain). Counting `True` values is how we will count combined pipes.

In [2]:
df = pd.read_csv("../data/vancouver_sewers_utm10n.csv")
points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["x"], df["y"]), crs="EPSG:26910")
points["is_combined"] = points["flow_type"] == "Combined"
print(len(points), "pipes,", points["is_combined"].sum(), "of them combined")

39713 pipes, 9656 of them combined


## Layer A, Part 2. Count the pipes in each square

A map of 39,713 dots shows where pipes are. It cannot show **where the old system still is**, because the dots of three kinds sit on top of each other. A summary can: cut the city into squares, count the pipes in each, and work out what share are combined. This is **aggregation**, and it is what you do whenever your question is "how many per area": trees per borough, potholes per street, collisions per neighbourhood.

Three steps, each one line of real work:

1. **Make the squares.** The function below is given: it lays a grid of 1,000 m squares over the pipes. (This is why we stayed in metres.)
2. **Spatial join.** For each pipe, find the square it falls in. `gpd.sjoin` is a join like a spreadsheet lookup, except the match is made by location instead of by a shared ID. Fill in the blank: we want squares that the pipe `"intersects"`.
3. **Group and count.** For each square: how many pipes, and how many combined. Fill in the blank with the word that adds up `True` values.

In [3]:
def make_grid(gdf, size):
    """Squares of `size` metres covering every feature in gdf (which must be in metres)."""
    minx, miny, maxx, maxy = gdf.total_bounds
    xs = np.arange(np.floor(minx / size) * size, maxx, size)
    ys = np.arange(np.floor(miny / size) * size, maxy, size)
    squares = [box(x, y, x + size, y + size) for x in xs for y in ys]
    return gpd.GeoDataFrame({"cell_id": range(len(squares))}, geometry=squares, crs=gdf.crs)

grid = make_grid(points, 1000)
print(len(grid), "squares of 1 km")

192 squares of 1 km


In [7]:
joined = gpd.sjoin(points, grid, predicate="intersects")       # which square does each pipe touch?
joined = joined[~joined.index.duplicated()]             # a pipe exactly on an edge touches two squares: keep it once

summary = joined.groupby("cell_id").agg(
    pipes=("pipe_id", "count"),        # how many pipes in the square
    combined=("is_combined", "sum")    # how many of them are combined: add up the True values
).reset_index()
summary["combined_pct"] = (100 * summary["combined"] / summary["pipes"]).round(0)

cells = grid.merge(summary, on="cell_id")   # keeps only squares that have pipes
print(len(cells), "squares have pipes")
print("Pipes counted:", cells["pipes"].sum(), "of", len(points))
cells.head()

142 squares have pipes
Pipes counted: 39713 of 39713


,cell_id,geometry,pipes,combined,combined_pct
0,8,"POLYGON ((484000 5457000, 484000 5458000, 4830...",15,8,53.0
1,9,"POLYGON ((484000 5458000, 484000 5459000, 4830...",11,6,55.0
2,16,"POLYGON ((485000 5453000, 485000 5454000, 4840...",4,0,0.0
3,19,"POLYGON ((485000 5456000, 485000 5457000, 4840...",179,23,13.0
4,20,"POLYGON ((485000 5457000, 485000 5458000, 4840...",338,90,27.0


**Check:** `Pipes counted` must equal the number of pipes. If it is smaller, some pipes fell into no square; if it is larger, some were counted twice (try the cell without the `duplicated` line to see it happen). A total that adds up is the cheapest test there is for any aggregation, so use it on your own data too.

**Look at the numbers before you choose colours.** Colours on a map are classes: you decide where one class ends and the next begins. Run the next cell and read the spread.

In [8]:
print(cells["combined_pct"].describe().round(1))
print()
print("Squares with no combined pipes:", (cells["combined_pct"] == 0).sum())
print("Squares that are at least half combined:", (cells["combined_pct"] >= 50).sum())

count    142.0
mean      27.3
std       24.6
min        0.0
25%        4.0
50%       20.5
75%       43.8
max      100.0
Name: combined_pct, dtype: float64

Squares with no combined pipes: 23
Squares that are at least half combined: 27


Most squares have a low share and a few are mostly combined. Five classes work here: **none**, **under 25%**, **25 to 49%**, **50 to 74%**, **75% or more**. The page's colour function uses exactly these breaks, and the legend must say the same thing.

## Layer A, Part 3. Write both layers, then check

The squares go to degrees for the web, with only the fields the popup needs. The second file is Lab 4's result again (the combined pipes as points), so the page has two layers to switch between.

In [9]:
web_cells = cells[["cell_id", "pipes", "combined", "combined_pct", "geometry"]].to_crs("EPSG:4326")
web_cells.to_file("../data/sewer_grid.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

combined_pipes = points[points["is_combined"]].to_crs("EPSG:4326")[["flow_type", "material", "length_m", "geometry"]]
combined_pipes.to_file("../data/combined_sewers.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

for f in ["../data/sewer_grid.geojson", "../data/combined_sewers.geojson"]:
    print(f, round(os.path.getsize(f) / 1_000_000, 2), "MB")

../data/sewer_grid.geojson 0.04 MB
../data/combined_sewers.geojson 1.81 MB


### The check cell (from Lab 4), now on the squares

It works for areas and lines as well as points. Change only the settings when you use it on your own files.

In [10]:
# ---- settings: change these for your own data ----
FILE = "../data/sewer_grid.geojson"
WEST, EAST, SOUTH, NORTH = -123.3, -122.9, 49.1, 49.4   # a box around your area, in degrees
EXPECTED = 142            # how many features you expect
POPUP_FIELDS = ["pipes", "combined_pct"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Polygon']
All checks passed


**Done with the Python half when** the check prints `All checks passed`. Now open `site/index.html` with Live Server and follow Part 4 in the handout: three changes in `site/app.js` (colour by value, legend, layer switcher).

---

# Layer B. Your second layer

Your second layer comes from one of three places. Use the matching recipe **in a copy of this notebook inside your own project** (`notebooks/`). Every recipe is commented out: select its lines, press **Ctrl+/** to remove the `#` marks, and change the names in capitals. Print your columns first with `print(gdf.columns.tolist())`.

| Your second layer is... | Use |
| --- | --- |
| A second dataset (parks, stations, boundaries, a second year) | Lab 4's recipes 1 to 4 to read and convert it, then the check cell |
| A summary of your own data per area (count per borough, per neighbourhood, per square) | Recipe 6 below, or the Layer A grid code |
| A value from a table without coordinates (a score, a count, a condition) | Recipe 7 below |
| How far each feature is from the nearest something | Recipe 8 below |
| How many of something are within a distance of each feature | Recipe 9 below |
| Your own data split by a category | No Python needed: do it on the page (handout B3) |

### Recipe 6. Count your features per area

You need an area file (boroughs, neighbourhoods, census areas) as well as your points. Both must be in the **same** coordinate system before the join; converting both to your city's metre system (Montréal: MTM zone 8, `EPSG:2950`) is safest. The total check at the end is the same idea as in Layer A.

In [ ]:
# pts = gpd.read_file("../data/YOUR_POINTS.geojson").to_crs("EPSG:2950")
# areas = gpd.read_file("../data/YOUR_AREAS.geojson").to_crs("EPSG:2950")
# areas = areas[["AREA_NAME_COLUMN", "geometry"]]
# j = gpd.sjoin(pts, areas, predicate="within")
# counts = j.groupby("AREA_NAME_COLUMN").size().reset_index(name="count")
# areas = areas.merge(counts, on="AREA_NAME_COLUMN", how="left").fillna({"count": 0})
# print("Counted", int(areas["count"].sum()), "of", len(pts), "points")   # a few outside every area is normal; many is not
# areas.to_crs("EPSG:4326").to_file("../data/YOUR_AREAS_WITH_COUNTS.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

### Recipe 7. Attach a table to your features by a shared ID

When the value you want lives in a second table with no coordinates (a condition score per street segment, a population per area), join on a column both tables share, like `ID_TRC` for Montréal street segments. This is an ordinary join, not a spatial one. Check how many matched: if hardly any did, the two ID columns are probably different types (text versus number) or formats.

In [ ]:
# gdf = gpd.read_file("../data/YOUR_FEATURES.geojson")
# table = pd.read_csv("../data/YOUR_TABLE.csv")
# print(gdf["ID_COLUMN"].dtype, table["ID_COLUMN"].dtype)        # must be the same type to match
# # If one says object (text) and the other int64 (a number), also remove the # from the next line:
# # table["ID_COLUMN"] = table["ID_COLUMN"].astype(str); gdf["ID_COLUMN"] = gdf["ID_COLUMN"].astype(str)
# gdf = gdf.merge(table[["ID_COLUMN", "VALUE_COLUMN"]], on="ID_COLUMN", how="left")
# print(gdf["VALUE_COLUMN"].notna().sum(), "of", len(gdf), "features matched")
# gdf.to_file("../data/YOUR_OUTPUT.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

### Recipe 8. Distance to the nearest something

For questions like "how far is each school from a park" or "which areas are far from a metro station". Distances must be measured in **metres**, so convert both layers to a metre system first; measured in degrees, a distance means nothing. This is straight-line distance, not walking distance: say so in your README.

In [ ]:
# a = gpd.read_file("../data/YOUR_FEATURES.geojson").to_crs("EPSG:2950")      # e.g. schools
# b = gpd.read_file("../data/YOUR_TARGETS.geojson").to_crs("EPSG:2950")       # e.g. parks
# a = gpd.sjoin_nearest(a, b[["TARGET_NAME_COLUMN", "geometry"]], distance_col="dist_m")
# a = a[~a.index.duplicated()]                                               # ties: keep one
# a["dist_m"] = a["dist_m"].round(0)
# print(a["dist_m"].describe())
# a.drop(columns="index_right").to_crs("EPSG:4326").to_file("../data/YOUR_OUTPUT.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

### Recipe 9. How many are within a distance

For "how many parks within 500 m of each school" or "how many stores within a 15-minute walk (about 1,200 m) of each station". Draw a circle of that radius around each feature (a **buffer**), then count what falls inside it with a spatial join, as in Layer A. The radius is in the file's units, so convert to metres first; a buffer of 500 in degrees would cover half the planet.

In [ ]:
# a = gpd.read_file("../data/YOUR_FEATURES.geojson").to_crs("EPSG:2950")    # e.g. schools
# b = gpd.read_file("../data/YOUR_TARGETS.geojson").to_crs("EPSG:2950")     # e.g. parks (points or areas)
# RADIUS = 500                                                              # metres
# zones = a[["ID_COLUMN", "geometry"]].copy()
# zones["geometry"] = zones.buffer(RADIUS)                                   # a 500 m circle around each feature
# j = gpd.sjoin(b, zones, predicate="intersects")                           # which targets touch which circle
# counts = j.groupby("ID_COLUMN").size().reset_index(name="within_500m")
# a = a.merge(counts, on="ID_COLUMN", how="left").fillna({"within_500m": 0})
# print(a["within_500m"].describe())
# a.to_crs("EPSG:4326").to_file("../data/YOUR_OUTPUT.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

**Then:** run the check cell above on your new file (your file, your box, your count, your popup fields), put the layer on your page (handout B3 and B4), and write its receipt in your README.